# CASMI 2026 training spectra: download and first look

Pilot for the overlap of the Enveda CASMI 2026 training structures with PlantMetWiki (plan in [../README.md](../README.md)). This notebook downloads the competition files and checks the columns. Nothing here has been run yet: the download needs the competition rules to be accepted on Kaggle.

## Citation

David Healey, Christoph Krettler, Tobias Kind, Marie Killian, Seth Drake, Erik DeBloois, Ivy Lightheart, Yojana Gadiya, Pelle Simpson, Daniel Domingo-Fernandez, Viswa Colluru, August Allen, Walter Reade, Ashley Oldacre. *Enveda CASMI 2026 - Molecule ID From Mass Spectra*. <https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra>, 2026. Kaggle.

Use of the data is under the competition rules; the raw files stay out of git (`data/` is git-ignored).

## Credentials

The Kaggle token is read from `../kaggle.env` (one line, `KAGGLE_API_TOKEN=...`). That file is git-ignored and must never be committed or printed.

In [ ]:
import os
from pathlib import Path
import pandas as pd

HERE = Path.cwd().parent                       # pilots/casmi26_overlap (notebook runs from notebooks/)
for line in (HERE / "kaggle.env").read_text().splitlines():
    if "=" in line:
        k, v = line.split("=", 1); os.environ[k.strip()] = v.strip()
os.environ["KAGGLEHUB_CACHE"] = str(HERE / "data")   # keep the files inside the (ignored) data folder
import kagglehub

In [ ]:
path = Path(kagglehub.competition_download("enveda-CASMI26-molecule-id-mass-spectra"))
print("Path to competition files:", path)
sorted(p.name for p in path.iterdir())

## Columns

The description lists 18 columns for `train.parquet`, including `inchikey`, `inchikey14`, `normalized_smiles`, `molecular_formula`, `ingest_lib` and `precursor_error_ppm`. Check this against the file.

In [ ]:
import pyarrow.parquet as pq
pf = pq.ParquetFile(path / "train.parquet")
print(pf.metadata.num_rows, "rows;", pf.metadata.num_columns, "columns")
print(pf.schema_arrow.names)

## Unique structures per library

Read only the columns needed (the peak arrays are large).

In [ ]:
tr = pd.read_parquet(path / "train.parquet", columns=["inchikey14", "ingest_lib", "precursor_error_ppm", "adduct"])
summary = tr.groupby("ingest_lib").agg(spectra=("inchikey14", "size"), structures=("inchikey14", "nunique")).sort_values("spectra", ascending=False)
summary.loc["all (unique inchikey14)"] = [len(tr), tr.inchikey14.nunique()]
summary

The next step (match `inchikey14` to PlantMetWiki and to its non-reaction pathways) is described in the README plan and not done yet.